# Diffusion Policy Push-T Training on Colab (Custom Implementation)

Train our custom U-Net 1D implementation on Push-T dataset.
Matches original repo architecture: conditional_unet1d + cosine schedule + EMA + DDIM.

**Setup:** Upload the `diffusion/` folder to Colab (left sidebar → folder icon → upload folder)

In [ ]:
# Check GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Install dependencies
!pip install einops diffusers zarr pygame-ce pymunk -q

# Verify diffusion folder exists
import os
print(f'diffusion/ exists: {os.path.exists("/content/diffusion")}')
if os.path.exists("/content/diffusion"):
    print(os.listdir("/content/diffusion"))

In [ ]:
# Download Push-T dataset
!mkdir -p /content/diffusion/data/pusht
!wget -q https://diffusion-policy.cs.columbia.edu/data/training/pusht.zip -O /tmp/pusht.zip
!unzip -q /tmp/pusht.zip -d /content/diffusion/data/pusht/
!ls -la /content/diffusion/data/pusht/

In [ ]:
# Train custom implementation (500 epochs)
%cd /content/diffusion
!python train_ddpm.py \
    --epochs 500 \
    --batch_size 256 \
    --device cuda \
    --use_ema \
    --lr_warmup_steps 500 \
    --zarr_path data/pusht/pusht_cchi_v7_replay.zarr

In [ ]:
# Evaluate after training
%cd /content/diffusion
!python evaluate.py \
    --num_episodes 50 \
    --device cuda \
    --num_inference_steps 100 \
    --use_ddim True

In [ ]:
# Visualize results
%cd /content/diffusion
!python visualize.py --device cuda